In [ ]:
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import statistics

In [ ]:
N_ITERS = 5
T_OFFSET = 69.8
T_DURATION = 90

T_REACH_OFFSET = 70
TARGET_DISCOVERY_RATE = 99

In [ ]:
def read_reachability(target: str, n_peers: int) -> list[list[tuple[float, float]]]:
    data = []
    for i in range(N_ITERS):
        with open(f'../continuous/{target}/{n_peers}/r_{i}.txt', 'r') as f:
            time_init = 0
            measurements = []
            
            for line in f:
                parts = line.strip().split()
                if parts[1] == 'N/A':
                    continue

                time = int(parts[0])
                if time_init == 0:
                    time_init = time
                execution_time = (time - time_init) / 1000 - T_REACH_OFFSET
                reachability = float(parts[1])

                if execution_time < 0:
                    continue
                
                measurements.append((execution_time, reachability * 100))
            data.append(measurements)
    return data

In [ ]:
def time_split_discovery_time(data: list[tuple[float, float]]) -> list[float]:
    result = []
    time_now = 0.1
    split_discovery_time = 0.0
    for time, reach in data:
        if time >= time_now + 10:
            time_now += 10
            if split_discovery_time == 0.0:
                result.append(10.0)
            else:
                result.append(split_discovery_time)
                
            split_discovery_time = 0.0
            continue

        if reach > TARGET_DISCOVERY_RATE and split_discovery_time == 0.0:
            split_discovery_time = time - time_now + 0.1
    return result

In [ ]:
def time_split_discovery_time_all_seeds(target: str, n_peers: int) -> list[float]:
    result = []
    for data in read_reachability(target, n_peers):
        discovery_times = time_split_discovery_time(data)
        result.extend(discovery_times)
            
    print(f'time {target} {n_peers} median: {statistics.median(result)}')
    return result

In [ ]:
def read_ifstat_timesum(target: str, n_peers: int) -> list[list[float]]:
    result = [] # on each seed, time_series total bandwidth usage. Time data omitted.
    for seed in range(N_ITERS):
        file_paths = [f'../continuous/{target}/{n_peers}/{seed}/ifstat_h{i+1}.log' for i in range(n_peers)]
        
        files_data = [] # list[list[float]]
        for file_path in file_paths:
            netusages = [] # on time-series, the host's bandwidth usage (kbps)
            with open(file_path, 'r') as f:
                # Skip first two title lines
                next(f, None)
                next(f, None)
                for line_no, line in enumerate(f):
                    execution_time = line_no / 10 - T_OFFSET
                    netusage = sum([float(v) for v in line.strip().split()])
                    
                    # skip before burst
                    if execution_time < 0.0:
                        continue

                    if execution_time > T_DURATION:
                        break
                    
                    netusages.append(netusage)
            files_data.append(netusages)

        combined_netuse = [] # on this run, time-series, total bandwidth usage.
        min_length = min([len(d) for d in files_data])
        for i in range(min_length):
            combined_netuse.append(sum(
                [file_data[i] for file_data in files_data]
            ))

        result.append(combined_netuse)
    return result

In [ ]:
def time_split_interval_netuse(banwidth_time_series: list[float]) -> list[float]:
    result = []
    for i in range(len(banwidth_time_series) // (10 * 10)):
        result.append(sum(
            [v * 0.1 for v in banwidth_time_series[10*10*i:10*10*(i+1)]]
        ) / 8 / 1000) # MB
    return result

In [ ]:
def time_split_interval_netuse_all_seeds(target: str, n_peers) -> list[float]:
    seeds_data = read_ifstat_timesum(target, n_peers)
    
    result = [] # total network usage on a time split, for every seed, every split.
    for seed_data in seeds_data:
        seed_interval_netuses = time_split_interval_netuse(seed_data)
        result.extend(seed_interval_netuses)
    
    print(f'netuse {target} {n_peers} median: {statistics.median(result)}')
    return result

In [ ]:
COLORS = ["#00265C", "#9C1B2D", "#C7A705"]

def draw_dual_grouped_boxplot(
    data_left: list[list[list[float]]],
    data_right: list[list[list[float]]],
    x_labels: list[str],
    legend_labels: list[str] = ("Item 1", "Item 2", "Item 3"),
    y_labels: tuple[str, str] = ("Value A", "Value B"),
    y_ranges: tuple[tuple, tuple] = (None, None),
):
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 6,
        "axes.titlesize": 6,
        "axes.labelsize": 6,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "legend.fontsize": 6,
        "figure.titlesize": 6,
    })

    n_groups = len(data_left)
    n_positions = len(x_labels)
    
    # Calculate box width and spacing
    box_width = 0.8 / n_groups
    group_offset = (np.arange(n_groups) - (n_groups - 1) / 2) * box_width * 1.1

    # Calculate x-axis label constants
    x_numeric = np.arange(n_positions)

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(3.9, 1.5))

    # --- Left plot ---
    for i, (group, label) in enumerate(zip(data_left, legend_labels)):
        positions = np.arange(n_positions) + group_offset[i]
        bp = ax_left.boxplot(
            group,
            positions=positions,
            widths=box_width * 0.9,
            patch_artist=True,
            flierprops=dict(
                marker='.',
                markerfacecolor=COLORS[i],
                markeredgecolor=COLORS[i],
                markersize=3,
                linestyle='none'
            ),
            medianprops=dict(color=COLORS[i], linewidth=1),
            boxprops=dict(facecolor=COLORS[i], edgecolor=COLORS[i], alpha=0.7),
            whiskerprops=dict(color=COLORS[i]),
            capprops=dict(color=COLORS[i]),
        )
        # Add legend entry
        ax_left.plot([], [], color=COLORS[i], linewidth=2, alpha=0.7, label=label)
    
    ax_left.set_ylabel(y_labels[0], labelpad=1)
    #ax_left.set_xlabel(bottom=False)
    ax_left.set_xticks(x_numeric)
    ax_left.set_xticklabels(x_labels)
    ax_left.grid(axis='both', linestyle='--', alpha=0.4)
    ax_left.legend(loc='upper left', frameon=False)

    # --- Right plot ---
    for i, group in enumerate(data_right):
        positions = np.arange(n_positions) + group_offset[i]
        bp = ax_right.boxplot(
            group,
            positions=positions,
            widths=box_width * 0.9,
            patch_artist=True,
            flierprops=dict(
                marker='.',
                markerfacecolor=COLORS[i],
                markeredgecolor=COLORS[i],
                markersize=3,
                linestyle='none'
            ),
            medianprops=dict(color=COLORS[i], linewidth=1),
            boxprops=dict(facecolor=COLORS[i], edgecolor=COLORS[i], alpha=0.7),
            whiskerprops=dict(color=COLORS[i]),
            capprops=dict(color=COLORS[i]),
        )

    ax_right.set_ylabel(y_labels[1], labelpad=1)
    ax_right.set_xticks(x_numeric)
    ax_right.set_xticklabels(x_labels)
    ax_right.grid(axis='both', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.savefig("continuous.jpg", dpi=600, bbox_inches="tight", pad_inches=0)

In [ ]:
TARGETS = ['client-server', 'dev-eval-trickle', 'dev-v2']
N_PEERS = [110, 130, 200]



rng = np.random.default_rng(42)
draw_dual_grouped_boxplot(
    data_left=[
        [
            time_split_interval_netuse_all_seeds(target, n_peers) 
            for n_peers in N_PEERS
        ]
        for target in TARGETS
    ],
    data_right=[
        [
            time_split_discovery_time_all_seeds(target, n_peers) 
            for n_peers in N_PEERS
        ]
        for target in TARGETS
    ],
    x_labels=['1%', '3%', '10%'],
    legend_labels=['Client-Server', 'Trickle', 'Ours'],
    y_labels=("Network (MB)", "Discovery Time (s)"),
)